In [1]:
install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

library(sqldf)
library(dplyr)
library(ggplot2)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
orders <- read.csv("orders.csv")
deliveries <- read.csv("deliveries.csv")
customers <- read.csv("customers.csv")
drivers <- read.csv("drivers.csv")
vehicles <- read.csv("vehicles.csv")
hubs <- read.csv("hubs.csv")
incidents <- read.csv("incidents.csv")
complaints <- read.csv("complaints.csv")
app_events <- read.csv("app_events.csv")

cat("All files loaded successfully")

All files loaded successfully

In [3]:
nrow(orders)
nrow(deliveries)

[1] 1250

[1] 950

In [5]:
clean_zone <- function(x) {
  tools::toTitleCase(trimws(tolower(x)))
}

orders$pickup_zone <- clean_zone(orders$pickup_zone)
orders$dropoff_zone <- clean_zone(orders$dropoff_zone)

cat("Zone names cleaned successfully")

Zone names cleaned successfully

In [6]:
query1 <- sqldf("
  SELECT
      o.pickup_zone,
      o.priority_level,
      COUNT(d.delivery_id) AS total_deliveries,
      SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
      ROUND(
        SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) * 100.0
        / COUNT(d.delivery_id),
      2) AS delay_rate_pct
  FROM orders o
  JOIN deliveries d
      ON o.order_id = d.order_id
  GROUP BY o.pickup_zone, o.priority_level
  HAVING total_deliveries >= 10
  ORDER BY delay_rate_pct DESC
  LIMIT 10
")

print(query1)

   pickup_zone priority_level total_deliveries delayed_deliveries
1          Ctr         Medium               20                 10
2          Ctr           High               28                 10
3          Ctr            Low               12                  4
4         East       Critical               10                  3
5      Airport           High               25                  7
6      Central           High               36                 10
7    Riverside         Medium               54                 15
8      Airport         Medium               44                 12
9      Central            Low               22                  6
10     Airport            Low               37                 10
   delay_rate_pct
1           50.00
2           35.71
3           33.33
4           30.00
5           28.00
6           27.78
7           27.78
8           27.27
9           27.27
10          27.03


In [7]:
query2 <- sqldf("
  SELECT
      dr.employment_type,
      COUNT(d.delivery_id) AS total_deliveries,
      SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
      ROUND(
        SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0
        / COUNT(d.delivery_id),
      2) AS failure_rate_pct,
      ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating,
      ROUND(AVG(d.manual_route_override_count), 2) AS avg_route_overrides
  FROM deliveries d
  JOIN drivers dr
      ON d.driver_id = dr.driver_id
  GROUP BY dr.employment_type
  ORDER BY failure_rate_pct DESC
")

print(query2)

  employment_type total_deliveries failed_deliveries failure_rate_pct
1        FullTime              582                85            14.60
2        Contract              126                18            14.29
3        PartTime              242                29            11.98
  avg_customer_rating avg_route_overrides
1                3.87                1.02
2                3.83                0.93
3                3.88                0.86


In [8]:
query3 <- sqldf("
  SELECT
      o.service_type,
      o.priority_level,
      COUNT(o.order_id) AS total_orders,
      ROUND(AVG(o.order_value), 2) AS avg_order_value,
      SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_orders,
      SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_orders,
      ROUND(
        (SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) +
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)) * 100.0
        / COUNT(o.order_id),
      2) AS problem_rate_pct
  FROM orders o
  JOIN deliveries d
      ON o.order_id = d.order_id
  GROUP BY o.service_type, o.priority_level
  HAVING total_orders >= 10
  ORDER BY problem_rate_pct DESC
  LIMIT 10
")

print(query3)

   service_type priority_level total_orders avg_order_value delayed_orders
1      Business         Medium           55          100.48             13
2       Medical           High           21           87.82              5
3       Medical         Medium           40           95.66              8
4        Retail            Low           57           83.44             15
5      Business            Low           26           95.60              7
6        Parcel         Medium           89           88.99             20
7     Passenger         Medium          105           93.44             27
8        Retail           High           52           84.45             10
9      Business           High           35           92.77              6
10       Retail         Medium           97           91.37             20
   failed_orders problem_rate_pct
1             15            50.91
2              4            42.86
3              8            40.00
4              7            38.60
5    

In [9]:
query4 <- sqldf("
  SELECT
      c.home_zone,
      cp.complaint_type,
      COUNT(cp.complaint_id) AS total_complaints,
      ROUND(AVG(cp.resolution_days), 2) AS avg_resolution_days,
      ROUND(SUM(cp.compensation_amount), 2) AS total_compensation,
      ROUND(AVG(cp.compensation_amount), 2) AS avg_compensation
  FROM complaints cp
  JOIN customers c
      ON cp.customer_id = c.customer_id
  GROUP BY c.home_zone, cp.complaint_type
  HAVING total_complaints >= 3
  ORDER BY total_compensation DESC
  LIMIT 10
")

print(query4)

   home_zone  complaint_type total_complaints avg_resolution_days
1      SOUTH    MissedPickup               11                6.91
2  RiverSide           Delay               10                7.20
3      NORTH           Delay                7                8.00
4       East    MissedPickup                6               13.33
5      North DriverBehaviour                5                7.40
6      NORTH        AppIssue                5                7.60
7    CENTRAL           Delay                8                6.50
8       EAST           Delay                8                7.88
9       West    MissedPickup                4                6.25
10     SOUTH DriverBehaviour                4               14.00
   total_compensation avg_compensation
1              289.06            26.28
2              211.25            23.47
3              189.45            27.06
4              158.27            26.38
5              146.53            29.31
6              144.59            28.92
7

In [10]:
query5 <- sqldf("
  SELECT
      zone_context,
      COUNT(event_id) AS total_app_events,
      SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) AS failed_app_events,
      ROUND(
        SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) * 100.0
        / COUNT(event_id),
      2) AS app_failure_rate_pct,
      ROUND(AVG(api_latency_ms), 2) AS avg_api_latency_ms
  FROM app_events
  WHERE zone_context IS NOT NULL
  GROUP BY zone_context
  HAVING total_app_events >= 10
  ORDER BY app_failure_rate_pct DESC, avg_api_latency_ms DESC
")

print(query5)

   zone_context total_app_events failed_app_events app_failure_rate_pct
1         North               29                 3                10.34
2         north               31                 3                 9.68
3         South               43                 4                 9.30
4     RiverSide               43                 4                 9.30
5     Riverside               44                 4                 9.09
6       Central               24                 2                 8.33
7       CENTRAL               26                 2                 7.69
8           Ctr               43                 3                 6.98
9       Airport               44                 3                 6.82
10         WEST               59                 4                 6.78
11         EAST               40                 2                 5.00
12        NORTH               33                 1                 3.03
13         West               35                 1              

In [11]:
# BEFORE OPTIMISATION: SQL scans the full app_events table first

slow_time <- system.time({
  slow_app_query <- sqldf("
    SELECT
        zone_context,
        COUNT(event_id) AS failed_slow_events,
        ROUND(AVG(api_latency_ms), 2) AS avg_latency_ms
    FROM app_events
    WHERE success_flag = 0
      AND api_latency_ms > 600
    GROUP BY zone_context
    ORDER BY failed_slow_events DESC
  ")
})

print(slow_app_query)
print(slow_time)

  zone_context failed_slow_events avg_latency_ms
1      Central                  2           1067
2         WEST                  1            862
3        South                  1            637
4    Riverside                  1           1138
5        North                  1            624
6          Ctr                  1            752
7      CENTRAL                  1            998
   user  system elapsed 
  0.036   0.007   0.045 


In [12]:
# AFTER OPTIMISATION: pre-filter the dataset in R before running SQL

problem_app_events <- app_events[
  app_events$success_flag == 0 & app_events$api_latency_ms > 600,
]

fast_time <- system.time({
  fast_app_query <- sqldf("
    SELECT
        zone_context,
        COUNT(event_id) AS failed_slow_events,
        ROUND(AVG(api_latency_ms), 2) AS avg_latency_ms
    FROM problem_app_events
    GROUP BY zone_context
    ORDER BY failed_slow_events DESC
  ")
})

print(fast_app_query)
print(fast_time)

cat("Original app event rows:", nrow(app_events), "\n")
cat("Rows after pre-filtering:", nrow(problem_app_events), "\n")
cat("Rows reduced by:", nrow(app_events) - nrow(problem_app_events), "\n")

  zone_context failed_slow_events avg_latency_ms
1      Central                  2           1067
2         WEST                  1            862
3        South                  1            637
4    Riverside                  1           1138
5        North                  1            624
6          Ctr                  1            752
7      CENTRAL                  1            998
   user  system elapsed 
  0.033   0.008   0.043 
Original app event rows: 640 
Rows after pre-filtering: 8 
Rows reduced by: 632 
